In [ ]:
import pandas as pd
import numpy as np

ranked = pd.read_csv('../outputs/model_output.csv')
industry = pd.read_csv('../outputs/cleaned_industry.csv')

print('Model output shape:', ranked.shape)
print('Columns:', ranked.columns.tolist())
print()
print('Trajectory distribution:')
print(ranked['Trajectory'].value_counts())
print()
print('Is_Taught_MSRIT distribution:')
print(ranked['Is_Taught_MSRIT'].value_counts())
ranked.head()

Model output shape: (95, 16)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Demand_Score', 'Label_MSRIT', 'Recommendation_Prob_MSRIT', 'Trajectory']

Trajectory distribution:
Trajectory
Transitional    51
Upcoming        26
Traditional     18
Name: count, dtype: int64

Is_Taught_MSRIT distribution:
Is_Taught_MSRIT
0    73
1    22
Name: count, dtype: int64


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Institute_Count,Institutional_Gap_Score,Company_Spread,Company_Spread_Norm,Velocity,Volatility,Demand_Score,Label_MSRIT,Recommendation_Prob_MSRIT,Trajectory
0,advanced sql,8,-4.000000,0.812500,1,1,3,0.4,8,0.8,-4.0,2.607681,0.9225,0,0.502247,Upcoming
1,api gateway,1,0.000000,0.250000,0,0,0,1.0,1,0.1,0.0,0.447214,0.3200,0,0.000000,Transitional
2,api rate limiting,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214,0.4700,0,0.000000,Transitional
3,auto scaling,1,0.000000,0.500000,0,0,0,1.0,1,0.1,0.0,0.447214,0.3700,0,0.000000,Transitional
4,aws,29,-1.571429,0.318966,0,1,5,0.0,10,1.0,-2.0,4.494441,6.8995,0,0.128687,Traditional


In [ ]:
try:
    import ruptures as rpt
    pelt_available = True
except ImportError:
    print('ruptures not installed. Run: pip install ruptures')
    pelt_available = False

def detect_pelt_transition(skill_df):
    """Detect change point year using PELT algorithm on yearly frequency."""
    yearly = (
        skill_df.groupby('Year').size()
        .reindex(range(2021, 2026), fill_value=0)
        .values
        .reshape(-1, 1)
        .astype(float)
    )
    if pelt_available:
        try:
            algo = rpt.Pelt(model='rbf').fit(yearly)
            bkpts = algo.predict(pen=1)
            if bkpts and bkpts[0] < 5:
                return 2021 + bkpts[0] - 1
        except Exception:
            pass

    yearly_flat = yearly.flatten()
    median_count = np.median(yearly_flat[yearly_flat > 0]) if any(yearly_flat > 0) else 0
    for i, count in enumerate(yearly_flat):
        if count >= median_count and count > 0:
            return 2021 + i
    return 2021

threshold_years = (
    industry.groupby('Skill')
    .apply(detect_pelt_transition, include_groups=False)
    .reset_index(name='Industry_Threshold_Year')
)

print('Industry threshold year sample:')
print(threshold_years.sort_values('Industry_Threshold_Year').head(15).to_string(index=False))
print()
print('Threshold year distribution:')
print(threshold_years['Industry_Threshold_Year'].value_counts().sort_index())

ruptures not installed. Run: pip install ruptures
Industry threshold year sample:
           Skill  Industry_Threshold_Year
         aws ec2                     2021
           ci/cd                     2021
 data structures                     2021
             css                     2021
           excel                     2021
      javascript                     2021
    kpi tracking                     2021
      networking                     2021
model deployment                     2021
  multithreading                     2021
           linux                     2021
  load balancing                     2021
            java                     2021
            html                     2021
   redis caching                     2021

Threshold year distribution:
Industry_Threshold_Year
2021    25
2022    21
2023    20
2024    18
2025    11
Name: count, dtype: int64


In [ ]:
min_year = industry['Year'].min()
max_year = industry['Year'].max()

ranked = ranked.merge(threshold_years, on='Skill', how='left')
ranked['Industry_Threshold_Year'] = ranked['Industry_Threshold_Year'].fillna(min_year).astype(int)

ranked['CLI'] = np.where(
    ranked['Is_Taught_MSRIT'] == 0,
    max_year - ranked['Industry_Threshold_Year'],
    0
).astype(int)

gap_mask = ranked['Is_Taught_MSRIT'] == 0
print('CLI distribution (gap skills only):')
print(ranked[gap_mask]['CLI'].value_counts().sort_index())
print()
print('Top 10 by CLI (longest unresolved gaps):')
print(
    ranked[gap_mask][['Skill','Trajectory','CLI','Freq','Company_Spread']]
    .sort_values('CLI', ascending=False)
    .head(10)
    .to_string(index=False)
)

CLI distribution (gap skills only):
CLI
0    11
1    16
2    16
3    13
4    17
Name: count, dtype: int64

Top 10 by CLI (longest unresolved gaps):
          Skill   Trajectory  CLI  Freq  Company_Spread
        aws ec2 Transitional    4     1               1
          ci/cd Transitional    4    10               7
 multithreading  Traditional    4     1               1
 load balancing Transitional    4     1               1
          linux Transitional    4    11               8
   kpi tracking  Traditional    4     1               1
          excel Transitional    4     6               6
            css Transitional    4    11               9
data structures Transitional    4     4               4
shell scripting Transitional    4     3               3


In [ ]:
median_volatility = ranked['Volatility'].median()
TOTAL_COMPANIES = industry['Company'].nunique()

watch_mask = (
    ((ranked['Upcoming_Flag'] == 1) | (ranked['Trajectory'] == 'Upcoming')) &
    (ranked['Velocity'] >= 0) &
    (ranked['Volatility'] < median_volatility) &
    (ranked['Is_Taught_MSRIT'] == 0)
)

ranked['Watch_List'] = watch_mask.astype(int)

ranked['Watch_Urgency'] = np.where(
    watch_mask,
    (ranked['Company_Spread'] / TOTAL_COMPANIES) * ranked['Recency_Weight'],
    0.0
).round(4)

print(f'Watch List size: {watch_mask.sum()} skills')
print('Watch List skills (sorted by urgency):')
print(
    ranked[watch_mask][
        ['Skill','Trajectory','Velocity','Volatility',
         'Company_Spread','Recency_Weight','Watch_Urgency']
    ].sort_values('Watch_Urgency', ascending=False).to_string(index=False)
)

Watch List size: 0 skills
Watch List skills (sorted by urgency):
Empty DataFrame
Columns: [Skill, Trajectory, Velocity, Volatility, Company_Spread, Recency_Weight, Watch_Urgency]
Index: []


In [ ]:
yearly_type = (
    industry.groupby(['Skill','Year'])['Skill_Type']
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .sort_values(['Skill','Year'])
)

def detect_transition(skill_df):
    types = skill_df['Skill_Type'].tolist()
    if len(types) < 2:
        return None
    if len(set(types)) > 1:
        return f"{types[0]} -> {types[-1]}"
    return None

transitions = (
    yearly_type.groupby('Skill')
    .apply(detect_transition, include_groups=False)
    .reset_index(name='Transition')
)
transitions = transitions[transitions['Transition'].notna()]

transitions = transitions.merge(threshold_years, on='Skill', how='left')

print(f'Skills with detected type transitions: {len(transitions)}')
print('Transition patterns:')
print(transitions['Transition'].value_counts().to_string())
print()
print('Transitioning skills with year:')
print(transitions[['Skill','Transition','Industry_Threshold_Year']].to_string(index=False))

# Merge back into ranked
ranked = ranked.merge(
    transitions[['Skill','Transition']], on='Skill', how='left'
)
ranked['In_Transition'] = ranked['Transition'].notna().astype(int)

Skills with detected type transitions: 6
Transition patterns:
Transition
Traditional -> Transitional    2
Transitional -> Upcoming       2
Upcoming -> Transitional       1
Transitional -> Traditional    1

Transitioning skills with year:
              Skill                  Transition  Industry_Threshold_Year
 data visualization Traditional -> Transitional                     2023
distributed systems    Transitional -> Upcoming                     2024
         kubernetes    Upcoming -> Transitional                     2023
      microservices Transitional -> Traditional                     2022
            next.js    Transitional -> Upcoming                     2022
          rest apis Traditional -> Transitional                     2022


In [ ]:
%pip install networkx

from itertools import combinations
from collections import defaultdict
import networkx as nx

job_skill_groups = (
    industry.groupby(['Company','Year','Role'])['Skill']
    .apply(list)
    .reset_index(name='Skills')
)

cooccurrence = defaultdict(int)
for _, row in job_skill_groups.iterrows():
    skills = list(set(row['Skills']))
    for a, b in combinations(sorted(skills), 2):
        cooccurrence[(a, b)] += 1

cooccurrence_df = pd.DataFrame(
    [(a, b, count) for (a, b), count in cooccurrence.items()],
    columns=['Skill_A','Skill_B','Co_Count']
).sort_values('Co_Count', ascending=False)

G = nx.Graph()
for _, row in cooccurrence_df.iterrows():
    G.add_edge(row['Skill_A'], row['Skill_B'], weight=row['Co_Count'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Total skill pairs: {len(cooccurrence_df)}')
print()
print('Top 15 co-occurring skill pairs:')
print(cooccurrence_df.head(15).to_string(index=False))

gap_skills_set = set(ranked[ranked['Is_Taught_MSRIT']==0]['Skill'].tolist())
priority_lookup = ranked.set_index('Skill')['Recommendation_Prob_MSRIT'].to_dict()

propagated_gaps = defaultdict(float)
for gap_skill in gap_skills_set:
    if gap_skill not in G:
        continue
    gap_priority = priority_lookup.get(gap_skill, 0)
    for neighbor in G.neighbors(gap_skill):
        edge_weight = G[gap_skill][neighbor]['weight']
        propagated_gaps[neighbor] += 0.3 * gap_priority * edge_weight

propagated_df = pd.DataFrame(
    [(skill, score) for skill, score in propagated_gaps.items()],
    columns=['Skill','Propagated_Gap_Score']
).sort_values('Propagated_Gap_Score', ascending=False)

print()
print('Top 10 skills with highest propagated gap influence:')
print(propagated_df.head(10).to_string(index=False))

ranked = ranked.merge(propagated_df, on='Skill', how='left')
ranked['Propagated_Gap_Score'] = ranked['Propagated_Gap_Score'].fillna(0.0).round(4)

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Graph: 95 nodes, 270 edges
Total skill pairs: 270

Top 15 co-occurring skill pairs:
           Skill_A       Skill_B  Co_Count
              java           sql        21
          power bi        python        14
    cloud security    kubernetes        14
     generative ai           llm        13
              java system design        12
        kubernetes     terraform        12
            golang    kubernetes        12
data visualization        python        12
               aws    kubernetes        11
               css    javascript        11
     microservices system design        11
     microservices     rest apis        11
     deep learning    tensorflow         9
             react    typescript         9
            golang microservices         9

Top 10 skills with highest propagated gap influence:
              Skill  Propagated_Gap_Score
         kubernetes             12.336217
                aws             10.759599
     cloud security              6.843602
      

In [ ]:
def get_top_cooccurring(skill, top_n=3):
    if skill not in G:
        return []
    neighbors = sorted(
        G[skill].items(),
        key=lambda x: x[1]['weight'],
        reverse=True
    )
    return [n for n, _ in neighbors[:top_n]]

print('Co-occurrence clusters for top gap skills:')
for skill in ranked[ranked['Is_Taught_MSRIT']==0]['Skill'].head(10).tolist():
    cluster = get_top_cooccurring(skill)
    print(f'  {skill:35s} -> {cluster}')

ranked['Top_Cooccurring'] = ranked['Skill'].apply(
    lambda s: ', '.join(get_top_cooccurring(s, top_n=3))
)

cooccurrence_df.to_csv('../outputs/skill_cooccurrence.csv', index=False)
print('Saved skill_cooccurrence.csv')

Co-occurrence clusters for top gap skills:
  api gateway                         -> ['grpc', 'oauth2']
  api rate limiting                   -> ['background job processing', 'golang', 'microservices']
  auto scaling                        -> ['infrastructure as code', 'terraform']
  aws ec2                             -> ['aws', 'linux', 'load balancing']
  background job processing           -> ['api rate limiting', 'golang', 'microservices']
  blue-green deployment               -> ['canary deployment', 'cloud security', 'kubernetes']
  canary deployment                   -> ['blue-green deployment', 'cloud security', 'kubernetes']
  ci/cd                               -> ['docker', 'aws', 'linux']
  ci/cd pipelines                     -> ['aws', 'ci/cd', 'docker']
  circuit breaker pattern             -> ['microservices', 'rest apis', 'event-driven architecture']
Saved skill_cooccurrence.csv


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

priority_features = [
    'CLI',
    'Velocity',
    'Company_Spread',
    'Recommendation_Prob_MSRIT',
    'Institutional_Gap_Score',
]

norm_vals = scaler.fit_transform(ranked[priority_features])
norm_df = pd.DataFrame(
    norm_vals,
    columns=[f'{c}_norm' for c in priority_features]
)
ranked = pd.concat([ranked.reset_index(drop=True), norm_df], axis=1)

ranked['Priority_Score'] = (
    ranked['CLI_norm']                         * 0.30 +
    ranked['Velocity_norm']                    * 0.25 +
    ranked['Company_Spread_norm']              * 0.20 +
    ranked['Recommendation_Prob_MSRIT_norm']   * 0.15 +
    ranked['Institutional_Gap_Score_norm']     * 0.10
).round(4)

gap_mask = ranked['Is_Taught_MSRIT'] == 0
print('Top 15 gap skills by Priority Score:')
print(
    ranked[gap_mask][
        ['Skill','Trajectory','CLI','Company_Spread',
         'Institutional_Gap_Score','Priority_Score']
    ]
    .sort_values('Priority_Score', ascending=False)
    .head(15)
    .to_string(index=False)
)

Top 15 gap skills by Priority Score:
        Skill   Trajectory  CLI  Company_Spread  Institutional_Gap_Score  Priority_Score
   tensorflow Transitional    2              10                      1.0          0.8288
        linux Transitional    4               8                      1.0          0.8275
          css Transitional    4               9                      1.0          0.7939
        ci/cd Transitional    4               7                      1.0          0.7679
   networking Transitional    4               8                      1.0          0.7605
       docker Transitional    3               8                      1.0          0.7556
     power bi Transitional    3               9                      1.0          0.7267
      next.js     Upcoming    3               7                      1.0          0.7138
   kubernetes Transitional    2              10                      0.8          0.7037
    terraform  Traditional    2               9                      1.0 

In [ ]:
max_vol = ranked['Volatility'].max()
ranked['Volatility_inv_norm'] = (1 - (ranked['Volatility'] / max_vol)).round(4)

vel_min = ranked['Velocity'].min()
vel_range = ranked['Velocity'].max() - vel_min
ranked['Velocity_pos'] = ((ranked['Velocity'] - vel_min) / vel_range).round(4) if vel_range != 0 else 0

ranked['Debt_Score'] = (
    ranked['CLI']                      *
    (1 + ranked['Velocity_pos'])       *
    (1 + ranked['Institutional_Gap_Score'])
).round(4)

ranked.loc[ranked['Is_Taught_MSRIT'] == 1, 'Debt_Score'] = 0.0

total_debt = ranked['Debt_Score'].sum()
print(f'Total Institutional Curriculum Debt Score: {total_debt:.2f}')
print()
print('Top 15 by Debt Score:')
print(
    ranked[gap_mask][
        ['Skill','Trajectory','CLI','Velocity_pos','Institutional_Gap_Score','Debt_Score']
    ]
    .sort_values('Debt_Score', ascending=False)
    .head(15)
    .to_string(index=False)
)

Total Institutional Curriculum Debt Score: 454.29

Top 15 by Debt Score:
               Skill   Trajectory  CLI  Velocity_pos  Institutional_Gap_Score  Debt_Score
             aws ec2 Transitional    4           0.5                      1.0        12.0
   vpc configuration Transitional    4           0.5                      1.0        12.0
      multithreading  Traditional    4           0.5                      1.0        12.0
        scikit-learn Transitional    4           0.5                      1.0        12.0
      load balancing Transitional    4           0.5                      1.0        12.0
        kpi tracking  Traditional    4           0.5                      1.0        12.0
               linux Transitional    4           0.5                      1.0        12.0
               excel Transitional    4           0.5                      1.0        12.0
     data structures Transitional    4           0.5                      1.0        12.0
     shell scripting Transi

In [ ]:
drop_cols = [c for c in ranked.columns if c.endswith('_norm') or c == 'Velocity_pos']
final = ranked.drop(columns=drop_cols)

final.to_csv('../outputs/skill_analysis.csv', index=False)
print('Saved skill_analysis.csv')
print('Shape:', final.shape)
print('Columns:', list(final.columns))
print()
print('=== SKILL ANALYSIS SUMMARY ===')
print(f'Total skills analysed      : {len(final)}')
print(f'Gap skills (not in MSRIT)  : {gap_mask.sum()}')
print(f'Watch List skills          : {final["Watch_List"].sum()}')
print(f'Skills in transition       : {final["In_Transition"].sum()}')
print(f'Avg CLI (gap skills)       : {final[gap_mask]["CLI"].mean():.1f} years')
print(f'Max CLI                    : {final[gap_mask]["CLI"].max()} years')
print(f'Total curriculum debt      : {final["Debt_Score"].sum():.2f}')
print()
print('Top 10 final priority gaps:')
print(
    final[gap_mask][
        ['Skill','Trajectory','Priority_Score','Debt_Score','Watch_List']
    ]
    .sort_values('Priority_Score', ascending=False)
    .head(10)
    .to_string(index=False)
)

Saved skill_analysis.csv
Shape: (95, 26)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Demand_Score', 'Label_MSRIT', 'Recommendation_Prob_MSRIT', 'Trajectory', 'Industry_Threshold_Year', 'CLI', 'Watch_List', 'Watch_Urgency', 'Transition', 'In_Transition', 'Propagated_Gap_Score', 'Top_Cooccurring', 'Priority_Score', 'Debt_Score']

=== SKILL ANALYSIS SUMMARY ===
Total skills analysed      : 95
Gap skills (not in MSRIT)  : 73
Watch List skills          : 0
Skills in transition       : 6
Avg CLI (gap skills)       : 2.1 years
Max CLI                    : 4 years
Total curriculum debt      : 454.29

Top 10 final priority gaps:
     Skill   Trajectory  Priority_Score  Debt_Score  Watch_List
tensorflow Transitional          0.8288      8.0000           0
     linux Transitional          0.8275     12.0000           0
       